# 🎯 Optimal Pricing Engine
---

## Optimization Problem

$$\max_{\mathbf{P}} \quad \Pi(\mathbf{P}) = \sum_{i=1}^{n} (P_i - C_i) \cdot Q_i(\mathbf{P})$$

**Subject to:**
- Margin: $(P_i - C_i)/P_i \geq 15\%$
- Price bounds: $P_i^0 (1-10\%) \leq P_i \leq P_i^0 (1+10\%)$

## Demand Function

Using elasticity matrix:

$$Q_i(\mathbf{P}) = Q_i^0 \cdot \exp\left( \sum_{j} \varepsilon_{ij} \cdot \ln\frac{P_j}{P_j^0} \right)$$

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
import statsmodels.api as sm
from statsmodels.regression.linear_model import OLS
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from snowflake.snowpark.context import get_active_session
session = get_active_session()

---
## 1. Build Elasticity Matrix

In [ ]:
# Estimate SUR (condensed)
fam_df = session.sql('''
    SELECT order_date, product_family, SUM(total_quantity_mt) q, AVG(avg_price_per_mt) p
    FROM CHEMICALS_DB.CHEMICAL_OPS.ML_TRAINING_DATA WHERE ln_quantity IS NOT NULL GROUP BY 1,2
''').to_pandas()

families = sorted(fam_df['PRODUCT_FAMILY'].unique().tolist())
fids = [f.replace(' ','_').upper() for f in families]

price_wide = fam_df.pivot(index='ORDER_DATE', columns='PRODUCT_FAMILY', values='p').reset_index()
qty_wide = fam_df.pivot(index='ORDER_DATE', columns='PRODUCT_FAMILY', values='q').reset_index()
sur = price_wide.merge(qty_wide, on='ORDER_DATE', suffixes=('_P','_Q'))
for f, fid in zip(families, fids):
    sur[f'LN_P_{fid}'] = np.log(sur[f'{f}_P'].clip(1))
    sur[f'LN_Q_{fid}'] = np.log(sur[f'{f}_Q'].clip(1))
sur = sur.dropna()

pcols = [f'LN_P_{f}' for f in fids]
E = pd.DataFrame(index=fids, columns=fids, dtype=float)
for fid in fids:
    y, X = sur[f'LN_Q_{fid}'], sm.add_constant(sur[pcols])
    m = OLS(y, X).fit()
    for pf in fids: E.loc[fid, pf] = m.params[f'LN_P_{pf}']

print('📊 ELASTICITY MATRIX')
print(E.round(3))

---
## 2. Current State

In [ ]:
cur = session.sql('''
    SELECT product_family, AVG(avg_price_per_mt) p, AVG(avg_variable_cost) c, SUM(total_quantity_mt) q
    FROM CHEMICALS_DB.CHEMICAL_OPS.ML_TRAINING_DATA
    WHERE order_date >= DATEADD(day,-30,CURRENT_DATE()) GROUP BY 1
''').to_pandas()
cur['FID'] = cur['PRODUCT_FAMILY'].str.replace(' ','_').str.upper()
cur = cur.set_index('FID').loc[fids]

P0 = cur['P'].values
Q0 = cur['Q'].values
C = cur['C'].values
E_np = E.values.astype(float)

print('📊 CURRENT STATE')
for i, f in enumerate(fids):
    margin = (P0[i]-C[i])/P0[i]*100
    print(f'   {f:15s}: P=${P0[i]:,.0f} C=${C[i]:,.0f} Margin={margin:.1f}%')

---
## 3. Optimization

In [ ]:
# Functions
def demand(P, Q0, E, P0):
    return Q0 * np.exp(E @ np.log(P/P0))

def profit(P, Q0, C, E, P0):
    return np.sum((P-C) * demand(P,Q0,E,P0))

def neg_profit(P, *args):
    return -profit(P, *args)

# Constraints
MIN_MARGIN, MAX_CHANGE = 0.15, 0.10
bounds = [(max(P0[i]*(1-MAX_CHANGE), C[i]/(1-MIN_MARGIN)), P0[i]*(1+MAX_CHANGE)) for i in range(len(P0))]

# Optimize
result = minimize(neg_profit, P0, args=(Q0,C,E_np,P0), method='SLSQP', bounds=bounds)

P_opt = result.x
Q_opt = demand(P_opt, Q0, E_np, P0)
cur_profit = profit(P0, Q0, C, E_np, P0)
opt_profit = profit(P_opt, Q0, C, E_np, P0)

print(f'\n🎯 OPTIMIZATION: {"✅" if result.success else "❌"}')
print(f'   Iterations: {result.nit}')

---
## 4. Results

In [ ]:
print('\n📊 OPTIMIZATION RESULTS')
print('=' * 80)
print(f'{"Product":<15} {"Current":>10} {"Optimal":>10} {"Δ Price":>10} {"Δ Volume":>10} {"Δ Margin":>12}')
print('-' * 80)

for i, f in enumerate(fids):
    dp = (P_opt[i]-P0[i])/P0[i]*100
    dq = (Q_opt[i]-Q0[i])/Q0[i]*100
    dm = (P_opt[i]-C[i])*Q_opt[i] - (P0[i]-C[i])*Q0[i]
    print(f'{f:<15} ${P0[i]:>9,.0f} ${P_opt[i]:>9,.0f} {dp:>+9.1f}% {dq:>+9.1f}% ${dm:>+11,.0f}')

print('-' * 80)
print(f'\n💰 PROFIT: ${cur_profit:,.0f} → ${opt_profit:,.0f} ({(opt_profit/cur_profit-1)*100:+.1f}%)')

In [ ]:
# Visualization
fig = make_subplots(rows=1, cols=2, subplot_titles=['Price Changes', 'Margin Waterfall'])

changes = [(P_opt[i]-P0[i])/P0[i]*100 for i in range(len(P0))]
colors = ['#22c55e' if c>=0 else '#ef4444' for c in changes]
fig.add_trace(go.Bar(x=fids, y=changes, marker_color=colors,
                     text=[f'{c:+.1f}%' for c in changes], textposition='outside'), row=1, col=1)

margin_d = [(P_opt[i]-C[i])*Q_opt[i]-(P0[i]-C[i])*Q0[i] for i in range(len(P0))]
fig.add_trace(go.Waterfall(
    x=fids+['TOTAL'], y=margin_d+[sum(margin_d)],
    measure=['relative']*len(margin_d)+['total'],
    text=[f'${x:+,.0f}' for x in margin_d+[sum(margin_d)]], textposition='outside',
    increasing={'marker':{'color':'#22c55e'}}, decreasing={'marker':{'color':'#ef4444'}},
    totals={'marker':{'color':'#3b82f6'}}
), row=1, col=2)

fig.update_layout(template='plotly_dark', height=400, showlegend=False,
                  title='<b>Optimization Results</b>')
fig.show()

---
## 5. Recommendations

In [ ]:
print('\n📋 RECOMMENDED ACTIONS')
print('=' * 50)
for i, f in enumerate(fids):
    chg = (P_opt[i]-P0[i])/P0[i]*100
    if chg > 2:
        print(f'   {f}: ↑ INCREASE by {chg:.1f}%')
    elif chg < -2:
        print(f'   {f}: ↓ DECREASE by {abs(chg):.1f}%')
    else:
        print(f'   {f}: → HOLD current price')

---
## ✅ Optimization Complete

**Outputs:**
- Optimal price vector P*
- Expected profit improvement
- Action recommendations